In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [1]:
import os, glob, json
from pathlib import Path
import numpy as np
from tokenizers import Tokenizer

DATA = Path("/kaggle/input/datasets/megs132/capstone-1")

for p in sorted(DATA.iterdir()):
    print(f"{p.stat().st_size/1e6:9.1f} MB  {p.name}")

WORK     = "/kaggle/working"
CKPT_DIR = f"{WORK}/ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)

TRAIN_BIN, VAL_BIN = DATA / "bpe32k_train.bin", DATA / "bpe32k_val.bin"
META_PATH, TOK_PATH = DATA / "bpe32k_meta.json", DATA / "bpe32k.json"
TEST_PATH = DATA / "test.json"
VAL_PROMPTS_PATH, VAL_REFS_PATH = DATA / "val_prompts.json", DATA / "val_refs.json"

for p in (TRAIN_BIN, VAL_BIN, META_PATH, TOK_PATH, TEST_PATH,
          VAL_PROMPTS_PATH, VAL_REFS_PATH):
    assert p.exists(), f"missing: {p}"

META = json.load(open(META_PATH))
VOCAB_SIZE = META["vocab_size"]
PAD_ID = META["special_token_ids"]["pad"]
EOS_ID = META["special_token_ids"]["eos"]

tok = Tokenizer.from_file(str(TOK_PATH))
train_bin = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_bin   = np.memmap(VAL_BIN,   dtype=np.uint16, mode="r")

test_data   = json.load(open(TEST_PATH))
VAL_PROMPTS = json.load(open(VAL_PROMPTS_PATH))
VAL_REFS    = json.load(open(VAL_REFS_PATH))

print(f"\nvocab {VOCAB_SIZE:,} | pad {PAD_ID} | eos {EOS_ID}")
print(f"train {len(train_bin):,} tokens | val {len(val_bin):,} tokens")
print(f"test {len(test_data)} | val prompts {len(VAL_PROMPTS)}")
print("\nsample:", tok.decode(train_bin[:60].astype(np.int64).tolist())[:300])

      0.0 MB  __huggingface_repos__.json
      2.3 MB  bpe32k.json
      0.0 MB  bpe32k_meta.json
    434.4 MB  bpe32k_train.bin
      5.1 MB  bpe32k_val.bin
      0.0 MB  corpus_stats.json
      3.0 MB  test.json
    181.6 MB  train_acts.jsonl
    367.1 MB  train_high_courts.jsonl
    296.2 MB  train_supreme_court.jsonl
    183.4 MB  train_tribunals.jsonl
      3.1 MB  val_acts.jsonl
      3.1 MB  val_high_courts.jsonl
      0.9 MB  val_prompts.json
      0.2 MB  val_refs.json
      3.0 MB  val_supreme_court.jsonl
      3.1 MB  val_tribunals.jsonl

vocab 32,000 | pad 0 | eos 3
train 217,201,090 tokens | val 2,560,200 tokens
test 1000 | val prompts 300

sample: Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 1984, provided for construction of five floors inclu


In [2]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DIM, N_HEADS, MAX_SEQ = 768, 12, 2048
BLOCK = 1024
print(f"device {DEVICE} | vocab {VOCAB_SIZE:,} | dim {DIM} | heads {N_HEADS}")

device cuda | vocab 32,000 | dim 768 | heads 12


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x = x.float()                                   # fp32 reduction
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x * rms).to(dtype) * self.weight


class SwiGLU(nn.Module):
    def __init__(self, dim, mult=4, multiple_of=64):
        super().__init__()
        hidden = int(dim * mult * 2 / 3)
        hidden = multiple_of * ((hidden + multiple_of - 1) // multiple_of)

        self.w1 = nn.Linear(dim, hidden, bias=False)    # gate branch
        self.w2 = nn.Linear(dim, hidden, bias=False)    # value branch
        self.w3 = nn.Linear(hidden, dim, bias=False)    # project back

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


# ---------------- tests ----------------

torch.manual_seed(42)
B, T, D = 4, 32, 768
x = torch.randn(B, T, D)

norm = RMSNorm(D)
y = norm(x)
rms = y.pow(2).mean(-1).sqrt()
assert y.shape == x.shape
assert torch.allclose(rms, torch.ones_like(rms), atol=1e-4)
print(f"RMSNorm   ok | out RMS {rms.mean():.6f} | params {sum(p.numel() for p in norm.parameters()):,}")

# fp16 stability: large activations must not overflow
xb = torch.randn(2, 8, D) * 300
assert torch.isfinite(norm(xb.half() if torch.cuda.is_available() else xb)).all()
print("RMSNorm   fp16 overflow test ok")

mlp = SwiGLU(D)
z = mlp(x)
assert z.shape == x.shape
print(f"SwiGLU    ok | hidden {mlp.w1.out_features} | params {sum(p.numel() for p in mlp.parameters()):,}")

RMSNorm   ok | out RMS 1.000000 | params 768
RMSNorm   fp16 overflow test ok
SwiGLU    ok | hidden 2048 | params 4,718,592


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class RoPE(nn.Module):
    """Contiguous-half rotary embeddings.
    x = [x1 | x2] -> [x1*cos - x2*sin, x2*cos + x1*sin]
    This convention MUST match the inference path exactly."""

    def __init__(self, head_dim, max_seq):
        super().__init__()
        assert head_dim % 2 == 0, "head_dim must be even for RoPE"
        self.head_dim, self.max_seq = head_dim, max_seq

        inv_freq = 1.0 / (10000 ** (
            torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_seq, dtype=torch.float32), inv_freq)

        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, x, pos0=0):
        """x: (B, H, T, head_dim). pos0 is the absolute position of x[..., 0, :],
        used by cached generation where T == 1."""
        dtype = x.dtype
        B, H, T, D = x.shape
        assert D == self.head_dim
        assert pos0 + T <= self.max_seq, f"position {pos0 + T} exceeds max_seq {self.max_seq}"

        half = D // 2
        x = x.float()
        x1, x2 = x[..., :half], x[..., half:]

        cos = self.cos[pos0:pos0 + T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[pos0:pos0 + T].unsqueeze(0).unsqueeze(0)

        return torch.cat([x1 * cos - x2 * sin,
                          x2 * cos + x1 * sin], dim=-1).to(dtype)


class CausalSelfAttention(nn.Module):
    def __init__(self, dim, n_heads, max_seq, rope=True):
        super().__init__()
        assert dim % n_heads == 0, "dim must be divisible by n_heads"

        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.qkv      = nn.Linear(dim, 3 * dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope     = RoPE(self.head_dim, max_seq) if rope else None

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)   # (B, H, T, hd)

        if self.rope is not None:
            q, k = self.rope(q), self.rope(k)            # never v

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.out_proj(out)


# ============================================================
# TESTS
# ============================================================

torch.manual_seed(42)
B, T, D, H = 2, 16, 768, 12

attn = CausalSelfAttention(D, H, max_seq=2048, rope=False)
x = torch.randn(B, T, D)
y = attn(x)
assert y.shape == x.shape and torch.isfinite(y).all()
print("attention  shape/finite ok")

# causality: perturbing position t must not affect positions < t
t = 7
x1 = torch.randn(B, T, D)
x2 = x1.clone()
x2[:, t, :] = torch.randn(B, D)
with torch.no_grad():
    y1, y2 = attn(x1), attn(x2)

assert torch.allclose(y1[:, :t], y2[:, :t], atol=1e-6), "LEAK: past positions changed"
assert not torch.allclose(y1[:, t:], y2[:, t:], atol=1e-6), "position t had no effect"
print("attention  causality ok")

# --- RoPE ---
hd, max_seq = 64, 2048
rope = RoPE(hd, max_seq)

q = torch.randn(2, 8, 32, hd)
assert torch.allclose(q.norm(dim=-1), rope(q).norm(dim=-1), atol=1e-5), "not a rotation"
print("rope       norm preservation ok")

# relative position: score depends only on (i - j), with DISTINCT q and k
qv, kv = torch.randn(hd), torch.randn(hd)
qt, kt = torch.zeros(1, 1, 16, hd), torch.zeros(1, 1, 16, hd)
qt[0, 0, 5], kt[0, 0, 3]   = qv, kv      # offset 2
qt[0, 0, 12], kt[0, 0, 10] = qv, kv      # offset 2
qr, kr = rope(qt), rope(kt)

d1 = (qr[0, 0, 5]  * kr[0, 0, 3]).sum()
d2 = (qr[0, 0, 12] * kr[0, 0, 10]).sum()
d3 = (qr[0, 0, 12] * kr[0, 0, 3]).sum()   # offset 9 — must differ

assert torch.allclose(d1, d2, atol=1e-5), f"not relative: {d1:.6f} vs {d2:.6f}"
assert not torch.allclose(d1, d3, atol=1e-3), "position has no effect at all"
print(f"rope       relative position ok  (off2 {d1:.4f}={d2:.4f}, off9 {d3:.4f})")

# pos0 offset must equal the same absolute position computed in one pass
full = rope(qt)
piece = rope(qt[:, :, 12:13], pos0=12)
assert torch.allclose(full[:, :, 12:13], piece, atol=1e-5), "pos0 offset broken"
print("rope       pos0 offset ok")

# --- combined ---
rattn = CausalSelfAttention(768, 12, max_seq=2048)
xr = torch.randn(2, 128, 768)
with torch.no_grad():
    yr = rattn(xr)
assert yr.shape == xr.shape and torch.isfinite(yr).all()
print(f"rope-attn  ok | params {sum(p.numel() for p in rattn.parameters()):,}")

attention  shape/finite ok
attention  causality ok
rope       norm preservation ok
rope       relative position ok  (off2 -0.3179=-0.3179, off9 -4.3299)
rope       pos0 offset ok
rope-attn  ok | params 2,359,296


In [5]:
class GatedLinearRecurrence(nn.Module):
    """h_t = a_t * h_{t-1} + i_t * v_t, computed chunkwise in parallel.
    Linear in sequence length; a genuinely different token mixer from attention."""

    def __init__(self, dim, conv_kernel=3, chunk_size=16):
        super().__init__()
        self.dim, self.chunk_size, self.conv_kernel = dim, chunk_size, conv_kernel

        self.in_proj = nn.Linear(dim, 4 * dim, bias=False)

        # causal depthwise conv: pad LEFT only, then trim the tail
        self.dwconv = nn.Conv1d(dim, dim, conv_kernel,
                                padding=conv_kernel - 1, groups=dim, bias=False)

        self.decay_proj = nn.Linear(dim, dim, bias=True)
        self.out_proj   = nn.Linear(dim, dim, bias=False)

    def _scan(self, log_a, u):
        """log_a, u: (B, T, D) fp32. Returns h: (B, T, D)."""
        B, T, D = u.shape
        L = self.chunk_size
        pad = (-T) % L
        if pad:
            log_a = F.pad(log_a, (0, 0, 0, pad))
            u     = F.pad(u,     (0, 0, 0, pad))
        n = (T + pad) // L

        log_a = log_a.view(B, n, L, D)
        u     = u.view(B, n, L, D)

        A = log_a.cumsum(2)                                   # (B, n, L, D)

        # decay matrix M[i, j] = exp(A_i - A_j) for j <= i, else 0.
        # exponent is always <= 0 -> underflows to 0, never overflows.
        d = A.unsqueeze(3) - A.unsqueeze(2)                   # (B, n, L, L, D)
        tri = torch.tril(torch.ones(L, L, device=u.device, dtype=torch.bool))
        d = d.masked_fill(~tri[None, None, :, :, None], float("-inf"))
        M = torch.exp(d)

        local = torch.einsum("bnijd,bnjd->bnid", M, u)        # within-chunk

        h, outs = u.new_zeros(B, D), []
        decay = torch.exp(A)                                  # carry-in weight
        for c in range(n):
            outs.append(local[:, c] + decay[:, c] * h[:, None])
            h = outs[-1][:, -1]
        return torch.cat(outs, 1)[:, :T]

    def forward(self, x):
        B, T, D = x.shape

        # causal: left-pad by k-1, drop the k-1 trailing outputs
        c = self.dwconv(x.transpose(1, 2))[..., :T].transpose(1, 2)

        v, i, _, o = self.in_proj(c).chunk(4, dim=-1)
        v, i, o = torch.tanh(v), torch.sigmoid(i), torch.sigmoid(o)

        # log a in [-1.0, -0.01]: a range of useful timescales, no total reset
        log_a = -0.01 - 0.99 * torch.sigmoid(self.decay_proj(c).float())

        h = self._scan(log_a, i.float() * v.float())
        return self.out_proj((h * o.float()).to(x.dtype))

In [6]:
torch.manual_seed(0)
D = 768
glr = GatedLinearRecurrence(D).to(DEVICE)

x = torch.randn(2, 1024, D, device=DEVICE)
with torch.no_grad():
    y = glr(x)
assert y.shape == x.shape and torch.isfinite(y).all()
print(f"glr shape/finite ok | params {sum(p.numel() for p in glr.parameters()):,}")

# CAUSALITY: perturbing position t must not change any output before t
t = 400
a_, b_ = x.clone(), x.clone()
b_[:, t] = torch.randn(2, D, device=DEVICE)
with torch.no_grad():
    ya, yb = glr(a_), glr(b_)

before = (ya[:, :t] - yb[:, :t]).abs().max().item()
after  = (ya[:, t:] - yb[:, t:]).abs().max().item()
print(f"causality: before {before:.3e} | after {after:.3e}")
assert before < 1e-5, "LEAK: conv or scan sees the future"
assert after > 1e-3, "perturbation had no effect"

# chunk size must not change the result
g2 = GatedLinearRecurrence(D, chunk_size=16).to(DEVICE)
g2.load_state_dict(glr.state_dict())
with torch.no_grad():
    assert torch.allclose(glr(x), g2(x), atol=1e-3), "chunk boundary bug"
print("glr chunk-invariance ok")

# fp32 stability over a long run
with torch.no_grad():
    assert torch.isfinite(glr(torch.randn(1, 1024, D, device=DEVICE) * 5)).all()
print("glr fp32 stability ok")

glr shape/finite ok | params 3,542,016
causality: before 0.000e+00 | after 1.775e-01
glr chunk-invariance ok
glr fp32 stability ok


In [7]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

USE_AMP = DEVICE == "cuda"
C_LAYERS, C_ATTN_EVERY = 12, 3


class GenericBlock(nn.Module):
    """Pre-norm residual block with an injected token mixer."""
    def __init__(self, dim, mixer):
        super().__init__()
        self.norm1, self.mixer = RMSNorm(dim), mixer
        self.norm2, self.mlp   = RMSNorm(dim), SwiGLU(dim)

    def forward(self, x):
        x = x + self.mixer(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class GenericLM(nn.Module):
    def __init__(self, mixer_factory, vocab_size=VOCAB_SIZE, dim=DIM,
                 n_layers=12, max_seq=MAX_SEQ):
        super().__init__()
        self.vocab_size, self.dim, self.max_seq = vocab_size, dim, max_seq

        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.blocks  = nn.ModuleList(
            [GenericBlock(dim, mixer_factory(i)) for i in range(n_layers)])
        self.norm_f  = RMSNorm(dim)
        self.head    = nn.Linear(dim, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight          # tie

        self.apply(self._init_weights)

        std = 0.02 / math.sqrt(2 * n_layers)
        for b in self.blocks:
            nn.init.normal_(b.mixer.out_proj.weight, 0.0, std)
            nn.init.normal_(b.mlp.w3.weight,         0.0, std)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        T = idx.size(1)
        if T > self.max_seq:
            raise ValueError(f"T={T} exceeds max_seq={self.max_seq}")

        x = self.tok_emb(idx)
        for b in self.blocks:
            x = b(x)
        logits = self.head(self.norm_f(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, self.vocab_size),
                                   targets.reshape(-1))
        return logits, loss


def count_unique_params(m):
    return sum(p.numel() for p in {id(p): p for p in m.parameters()}.values())


def c_mixer_factory(i):
    """Attention every 3rd layer; gated linear recurrence elsewhere."""
    if (i + 1) % C_ATTN_EVERY == 0:
        return CausalSelfAttention(DIM, N_HEADS, max_seq=MAX_SEQ)
    return GatedLinearRecurrence(DIM, conv_kernel=3, chunk_size=16)


# ============================================================
# CHECKS
# ============================================================

torch.manual_seed(42)
model_c = GenericLM(c_mixer_factory, n_layers=C_LAYERS).to(DEVICE)

# ---- 1. parameter count ----
C_PARAMS = count_unique_params(model_c)
n_attn = sum(1 for i in range(C_LAYERS) if (i + 1) % C_ATTN_EVERY == 0)

print(f"params {C_PARAMS:,} ({C_PARAMS/1e6:.2f}M)")
assert 50_000_000 <= C_PARAMS <= 300_000_000, f"{C_PARAMS:,} outside 50M-300M"
print(f"  {n_attn} attention layers, {C_LAYERS - n_attn} recurrence layers")
print(f"  embedding (tied) {model_c.tok_emb.weight.numel():,}")

# ---- 2. initial loss ----
torch.manual_seed(42)
x = torch.randint(0, VOCAB_SIZE, (2, 512), device=DEVICE)
y = torch.randint(0, VOCAB_SIZE, (2, 512), device=DEVICE)

model_c.eval()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
    _, init_loss = model_c(x, y)
init_loss = init_loss.item()

print(f"\ninitial loss {init_loss:.4f} vs ln(vocab) {math.log(VOCAB_SIZE):.4f}")
assert 9.0 < init_loss < 12.0, f"unexpected init loss {init_loss:.4f}"

# ---- 3. causality: perturb a LATE token, earlier outputs must not move ----
torch.manual_seed(123)
xa = torch.randint(0, VOCAB_SIZE, (1, 256), device=DEVICE)
xb = xa.clone()
t = 200
xb[0, t] = (xa[0, t].item() + 1) % VOCAB_SIZE

with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
    la, _ = model_c(xa)
    lb, _ = model_c(xb)

before = (la[:, :t].float() - lb[:, :t].float()).abs().max().item()
after  = (la[:, t:].float() - lb[:, t:].float()).abs().max().item()
print(f"\ncausality: max delta before t {before:.3e} | from t {after:.3e}")
assert before < 1e-4, "LEAK: future token affected past positions"
assert after > 1e-3, "perturbation had no effect"

# ---- 4. overfit one real batch ----
model_c.train()
rng = np.random.RandomState(12345)
off = rng.randint(0, len(train_bin) - BLOCK - 1, size=2)

xo = torch.from_numpy(
    np.stack([train_bin[i:i + BLOCK] for i in off]).astype(np.int64)).to(DEVICE)
yo = torch.from_numpy(
    np.stack([train_bin[i + 1:i + 1 + BLOCK] for i in off]).astype(np.int64)).to(DEVICE)

opt = torch.optim.AdamW(model_c.parameters(), lr=1e-3, weight_decay=0.0)
sc  = torch.amp.GradScaler("cuda", enabled=USE_AMP)
hist = []

for step in range(100):
    opt.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
        _, loss = model_c(xo, yo)
    sc.scale(loss).backward()
    sc.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model_c.parameters(), 1.0)
    sc.step(opt); sc.update()
    hist.append(loss.item())
    if step == 0 or (step + 1) % 20 == 0:
        print(f"  overfit {step+1:3d}/100  loss {loss.item():.4f}")

print(f"\noverfit {hist[0]:.4f} -> {hist[-1]:.4f}")
assert hist[-1] < 1.0, f"gradients not flowing: {hist[-1]:.4f}"

del opt, sc, xo, yo
torch.cuda.empty_cache()

print(f"\nMODEL C READY — {C_PARAMS/1e6:.2f}M params "
      f"({100*(C_PARAMS-109_529_856)/109_529_856:+.1f}% vs A)")
print(f"hybrid: recurrence in {C_LAYERS-n_attn} layers, attention every {C_ATTN_EVERY}")

params 118,991,616 (118.99M)
  4 attention layers, 8 recurrence layers
  embedding (tied) 24,576,000

initial loss 10.5311 vs ln(vocab) 10.3735

causality: max delta before t 0.000e+00 | from t 3.363e+00
  overfit   1/100  loss 10.5144
  overfit  20/100  loss 4.8292
  overfit  40/100  loss 1.1251
  overfit  60/100  loss 0.0205
  overfit  80/100  loss 0.0010
  overfit 100/100  loss 0.0004

overfit 10.5144 -> 0.0004

MODEL C READY — 118.99M params (+8.6% vs A)
hybrid: recurrence in 8 layers, attention every 3


In [8]:
import os, time, math, random
import numpy as np
import torch

BLOCK, MICRO_BS, ACCUM = 1024, 4, 8
TOKENS_PER_STEP = MICRO_BS * ACCUM * BLOCK

LR, MIN_LR, WARMUP, WD = 6e-4, 6e-5, 200, 0.1

MAX_STEPS = 1800                      # identical to A and B — matched budget
EVAL_EVERY, SAVE_EVERY, LOG_EVERY = 250, 250, 50
VAL_BATCHES, SEED = 20, 42

CKPT_PATH    = "/kaggle/working/ckpt/C_hybrid_last.pt"
WEIGHTS_PATH = "/kaggle/working/ckpt/C_hybrid.pt"
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)

USE_AMP = DEVICE == "cuda"

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# fresh model — the overfit check left model_c memorised on 2 sequences
model_c = GenericLM(c_mixer_factory, n_layers=C_LAYERS).to(DEVICE)

train_data = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
val_data   = np.memmap(VAL_BIN,   dtype=np.uint16, mode="r")

batch_rng = np.random.RandomState(SEED + 1)


def get_batch(data, rng, bs):
    ix = rng.randint(0, len(data) - BLOCK - 1, size=bs)
    x = np.stack([data[i:i + BLOCK]         for i in ix]).astype(np.int64)
    y = np.stack([data[i + 1:i + 1 + BLOCK] for i in ix]).astype(np.int64)
    return (torch.from_numpy(x).to(DEVICE, non_blocking=True),
            torch.from_numpy(y).to(DEVICE, non_blocking=True))


def make_val_batches():
    """Same seed as A and B -> identical windows, comparable val losses."""
    rng, out = np.random.RandomState(SEED + 2), []
    for _ in range(VAL_BATCHES):
        ix = rng.randint(0, len(val_data) - BLOCK - 1, size=MICRO_BS)
        x = np.stack([val_data[i:i + BLOCK]         for i in ix]).astype(np.int64)
        y = np.stack([val_data[i + 1:i + 1 + BLOCK] for i in ix]).astype(np.int64)
        out.append((torch.from_numpy(x), torch.from_numpy(y)))
    return out


val_batches = make_val_batches()

decay   = [p for p in model_c.parameters() if p.requires_grad and p.dim() >= 2]
nodecay = [p for p in model_c.parameters() if p.requires_grad and p.dim() < 2]

optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": WD},
     {"params": nodecay, "weight_decay": 0.0}],
    lr=LR, betas=(0.9, 0.95), fused=(DEVICE == "cuda"))

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def get_lr(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    p = min(max((step - WARMUP) / (MAX_STEPS - WARMUP), 0.0), 1.0)
    return MIN_LR + (LR - MIN_LR) * 0.5 * (1.0 + math.cos(math.pi * p))


def save_checkpoint(step, hist):
    state = dict(model=model_c.state_dict(), optimizer=optimizer.state_dict(),
                 scaler=scaler.state_dict(), step=step, loss_history=hist,
                 batch_rng_state=batch_rng.get_state(),
                 torch_rng_state=torch.get_rng_state(),
                 cuda_rng_state=(torch.cuda.get_rng_state_all()
                                 if torch.cuda.is_available() else None),
                 config=dict(arch="glr-attention hybrid", n_layers=C_LAYERS,
                             attn_every=C_ATTN_EVERY, chunk_size=32,
                             block=BLOCK, micro_bs=MICRO_BS, accum=ACCUM,
                             lr=LR, min_lr=MIN_LR, warmup=WARMUP,
                             wd=WD, max_steps=MAX_STEPS))
    torch.save(state, CKPT_PATH + ".tmp")
    os.replace(CKPT_PATH + ".tmp", CKPT_PATH)

    fp16 = {k: v.detach().cpu().half() if v.is_floating_point() else v.detach().cpu()
            for k, v in model_c.state_dict().items()}
    torch.save(fp16, WEIGHTS_PATH + ".tmp")
    os.replace(WEIGHTS_PATH + ".tmp", WEIGHTS_PATH)


start_step, loss_history = 0, []

if os.path.exists(CKPT_PATH):
    ck = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    model_c.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    start_step   = ck["step"] + 1
    loss_history = ck.get("loss_history", [])
    batch_rng.set_state(ck["batch_rng_state"])
    torch.set_rng_state(ck["torch_rng_state"].cpu()
                        if torch.is_tensor(ck["torch_rng_state"]) else ck["torch_rng_state"])
    if torch.cuda.is_available() and ck.get("cuda_rng_state") is not None:
        torch.cuda.set_rng_state_all(ck["cuda_rng_state"])
    print(f"resumed from step {start_step}", flush=True)
else:
    print("starting fresh", flush=True)


@torch.no_grad()
def evaluate():
    model_c.eval()
    total = 0.0
    for xc, yc in val_batches:
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            _, l = model_c(xc.to(DEVICE), yc.to(DEVICE))
        total += l.item()
    model_c.train()
    return total / len(val_batches)


model_c.train()
print(f"device {DEVICE} | tokens/step {TOKENS_PER_STEP:,} | max_steps {MAX_STEPS} | "
      f"{C_LAYERS} layers, attention every {C_ATTN_EVERY}\n", flush=True)

warmup_done_t, warmup_done_step = None, start_step + 3
t_run = time.time()

for step in range(start_step, MAX_STEPS):
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    accum = torch.zeros((), device=DEVICE)

    for _ in range(ACCUM):
        x, y = get_batch(train_data, batch_rng, MICRO_BS)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            _, loss = model_c(x, y)
        scaler.scale(loss / ACCUM).backward()
        accum += loss.detach()

    scaler.unscale_(optimizer)
    gnorm = torch.nn.utils.clip_grad_norm_(model_c.parameters(), 1.0)
    scaler.step(optimizer); scaler.update()

    step_loss = (accum / ACCUM).item()
    loss_history.append(step_loss)

    if step == warmup_done_step:
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        warmup_done_t = time.time()

    if (step + 1) % LOG_EVERY == 0:
        if warmup_done_t and step > warmup_done_step:
            tps = ((step - warmup_done_step) * TOKENS_PER_STEP
                   / (time.time() - warmup_done_t))
            eta = (MAX_STEPS - step - 1) * TOKENS_PER_STEP / tps / 3600
            tps_s, eta_s = f"{tps:,.0f}", f"{eta:4.2f}h"
        else:
            tps_s, eta_s = "warmup", "  -  "
        print(f"step {step+1:6d}/{MAX_STEPS} | loss {step_loss:7.4f} | lr {lr:.2e} | "
              f"grad {float(gnorm):6.3f} | tok/s {tps_s} | "
              f"{(time.time()-t_run)/60:6.1f} min | eta {eta_s}", flush=True)

    if (step + 1) % EVAL_EVERY == 0:
        vl = evaluate()
        print(f"  >> val loss {vl:.4f} (ppl {math.exp(vl):.1f})", flush=True)

    if (step + 1) % SAVE_EVERY == 0 or step + 1 == MAX_STEPS:
        save_checkpoint(step, loss_history)
        print(f"  >> checkpoint @ step {step+1}", flush=True)

print(f"\nTRAINING COMPLETE — {step+1} steps, "
      f"{(step+1)*TOKENS_PER_STEP/1e6:.0f}M tokens, "
      f"{(time.time()-t_run)/3600:.2f} h", flush=True)
print(f"final train loss {step_loss:.4f} | final val loss {evaluate():.4f}", flush=True)

starting fresh
device cuda | tokens/step 32,768 | max_steps 6600 | 12 layers, attention every 3



OutOfMemoryError: CUDA out of memory. Tried to allocate 768.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 280.81 MiB is free. Including non-PyTorch memory, this process has 14.29 GiB memory in use. Of the allocated memory 13.44 GiB is allocated by PyTorch, and 726.32 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)